In [1]:
import numpy as np
import re
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import PauliEvolutionGate, QAOAAnsatz
from qiskit.quantum_info import SparsePauliOp

from qiskit.transpiler import Layout
from qiskit.transpiler.passes import LayoutTransformation
from qiskit.converters import dag_to_circuit, circuit_to_dag

from qiskit_aer import AerSimulator
from qiskit_aer.backends.backendconfiguration import AerBackendConfiguration

from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples


In [2]:
num_physical_qubits = 4
extended_swap_strat = ExtendedSwapStrategy.from_line(list(range(num_physical_qubits)), num_swap_layers=num_physical_qubits)

In [3]:
basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "cx", "swap", "h", "rx"]

backend_options = dict(
    method='unitary',
    device='CPU',
    precision='single',
    basis_gates=basis_gates
)


config = AerSimulator._DEFAULT_CONFIGURATION
config["n_qubits"] = num_physical_qubits
config["basis_gates"] = basis_gates
config = AerBackendConfiguration.from_dict(config)
backend = AerSimulator(configuration=config, coupling_map=extended_swap_strat._coupling_map, **backend_options) # 
backend.set_option("n_qubits", num_physical_qubits)
print(backend.configuration().to_dict()["n_qubits"])

4


In [4]:
edge_maps = {
    0: {0:2, 1:1, 2:0, 3:3},
    1: {0:1, 1:0, 2:2, 3:3}
}

In [2]:
h1 = SparsePauliOp(['IIZZ', 'ZIZI'], np.array([0.5, 1]))
h2 = SparsePauliOp(['IIZI', 'ZIZI'], np.array([2, 4]))

In [5]:
donor_qc = QuantumCircuit(4)
edge_map = {0:2, 1:3, 2:0, 3:1}
layout = Layout({donor_qc.qubits[key]: val for key, val in edge_map.items()})
h1.apply_layout([2, 3, 0, 1], 4)

SparsePauliOp(['ZZII', 'ZIZI'],
              coeffs=[0.5+0.j, 1. +0.j])

In [6]:
h1.apply_layout([layout.get_virtual_bits()[donor_qc.qubits[i]] for i in range(4)])

SparsePauliOp(['ZZII', 'ZIZI'],
              coeffs=[0.5+0.j, 1. +0.j])

In [9]:
new_cost_circ = QuantumCircuit(4)
new_cost_circ.append(PauliEvolutionGate(h1), [layout.get_virtual_bits()[donor_qc.qubits[i]] for i in range(4)])
new_cost_circ.decompose().draw()

q_0: ────────────────
                     
q_1: ─────────■──────
              │      
q_2: ─■───────┼──────
      │ZZ(1)  │ZZ(2) 
q_3: ─■───────■──────

NameError: name 'new_cost_circ' is not defined

In [72]:
donor_qc = QuantumCircuit(num_physical_qubits)
qc = QuantumCircuit(0, num_physical_qubits)
qc.add_register([donor_qc.qubits[edge_maps[0][i]] for i in range(num_physical_qubits)])

ham_circ = QuantumCircuit(num_physical_qubits)
ham_circ.append(PauliEvolutionGate(h1, time=1), [edge_maps[0][i] for i in range(num_physical_qubits)])
tham_circ = transpile(ham_circ, backend, optimization_level=0)

qc.compose(tham_circ, range(num_physical_qubits), inplace=True)
qc.barrier()


from_layout = Layout({i: donor_qc.qubits[edge_maps[0][i]] for i in range(num_physical_qubits)})
for instruction in tham_circ.data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            print(f'Swapping {int(matches[0]), int(matches[1])}')
            from_layout.swap(int(matches[0]), int(matches[1]))
        else:
            raise Exception('Did not find 2 swap indices')


to_layout = Layout({i: donor_qc.qubits[edge_maps[1][i]] for i in range(num_physical_qubits)})
transformation_pass = LayoutTransformation(extended_swap_strat._coupling_map, from_layout, to_layout)
# transformation_pass = LayoutTransformation(None, from_layout, to_layout)

print(f'From: {from_layout.get_physical_bits()}')
print(f'To: {to_layout.get_physical_bits()}')

swap_qc = QuantumCircuit(num_physical_qubits)
swap_qc = dag_to_circuit(transformation_pass.run(circuit_to_dag(swap_qc)))
qc.compose(swap_qc, range(num_physical_qubits), inplace=True)
qc.barrier()

ham_circ2 = QuantumCircuit(num_physical_qubits)
ham_circ2.append(PauliEvolutionGate(h2, time=1), [edge_maps[1][i] for i in range(num_physical_qubits)])
tham_circ2 = transpile(ham_circ2, backend, optimization_level=0)
qc.compose(tham_circ2, range(num_physical_qubits), inplace=True)
qc.barrier()


from_layout = Layout({i: donor_qc.qubits[edge_maps[1][i]] for i in range(num_physical_qubits)})
for instruction in tham_circ2.data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            print(f'Swapping {int(matches[0]), int(matches[1])}')
            from_layout.swap(int(matches[0]), int(matches[1]))
        else:
            raise Exception('Did not find 2 swap indices')


to_layout = Layout({i: donor_qc.qubits[i] for i in range(num_physical_qubits)})
transformation_pass = LayoutTransformation(None, from_layout, to_layout)
swap_qc = QuantumCircuit(num_physical_qubits)
swap_qc = dag_to_circuit(transformation_pass.run(circuit_to_dag(swap_qc)))
qc.compose(swap_qc, range(num_physical_qubits), inplace=True)


qc.save_unitary()

Swapping (3, 2)
From: {0: <Qubit register=(4, "q"), index=2>, 1: <Qubit register=(4, "q"), index=1>, 3: <Qubit register=(4, "q"), index=0>, 2: <Qubit register=(4, "q"), index=3>}
To: {0: <Qubit register=(4, "q"), index=1>, 1: <Qubit register=(4, "q"), index=0>, 2: <Qubit register=(4, "q"), index=2>, 3: <Qubit register=(4, "q"), index=3>}
Swapping (0, 1)
Swapping (3, 2)


In [76]:
qc.draw(fold=-1)

░        ░ ┌───────┐            ░     unitary 
  0: ────────────────────░──X─────░─┤ Rz(4) ├─X──────────░────────░────
                         ░  │     ░ └───────┘ │          ░        ░    
  1: ─■──────────■───────░──X──X──░───────────X──■───────░────────░────
      │ZZ(1)     │ZZ(2)  ░     │  ░              │ZZ(8)  ░        ░    
  2: ─■───────X──■───────░──X──X──░─────X────────■───────░──X─────░────
              │          ░  │     ░     │                ░  │     ░    
  3: ─────────X──────────░──X─────░─────X────────────────░──X─────░────
                         ░        ░                      ░        ░    
c: 4/══════════════════════════════════════════════════════════════════

In [ ]:
h1 = SparsePauliOp(['IIZZ', 'ZIZI'], np.array([0.5, 1]))
h2 = SparsePauliOp(['IIZI', 'ZIZI'], np.array([2, 4]))

In [78]:
edge_maps = {
    0: {0:2, 1:1, 2:0, 3:3},
    1: {0:1, 1:0, 2:2, 3:3}
}

In [79]:
default_qaoa = QAOAAnsatz(h1+h2, reps=1, initial_state=QuantumCircuit(num_physical_qubits))
tdefault_qaoa = transpile(default_qaoa, backend, optimization_level=0)

final_layout = Layout({i: tdefault_qaoa.qubits[i] for i in range(num_physical_qubits)})
for instruction in tdefault_qaoa.data:
    if instruction.operation.name == 'swap':
        qubits_str = str(instruction.qubits)
        matches = re.findall('index=([0-9]+)', qubits_str)
        if len(matches) == 2:
            final_layout.swap(int(matches[0]), int(matches[1]))
        else:
            raise Exception('Did not find 2 swap indices')

to_layout = Layout({i: tdefault_qaoa.qubits[i] for i in range(num_physical_qubits)})
transformation_pass = LayoutTransformation(extended_swap_strat._coupling_map, final_layout, to_layout)
swap_qc = QuantumCircuit(num_physical_qubits)
swap_qc = dag_to_circuit(transformation_pass.run(circuit_to_dag(swap_qc)))
tdefault_qaoa.compose(swap_qc, range(num_physical_qubits), inplace=True)

tdefault_qaoa.save_unitary()
params = [0, 1]
bdefault_qaoa = tdefault_qaoa.assign_parameters({param: params[idx] for idx, param in enumerate(tdefault_qaoa.parameters)})

In [80]:
bdefault_qaoa.draw(fold=-1)

┌───────┐                                      unitary 
q_0 -> 0 ──■──────┤ Rx(0) ├─────────────────────────────────────────░────
           │ZZ(1) └───────┘        ┌───────┐        ┌───────┐       ░    
q_1 -> 1 ──■────────────────■──────┤ Rz(4) ├─■──────┤ Rx(0) ├───────░────
         ┌───────┐          │ZZ(2) └───────┘ │ZZ(8) ├───────┤       ░    
q_2 -> 2 ┤ Rx(0) ├────X─────■────────────────■──────┤ Rx(0) ├─X─────░────
         └───────┘    │                             └───────┘ │     ░    
q_3 -> 3 ─────────────X───────────────────────────────────────X─────░────
                                                                    ░

In [ ]:
res_custom = backend.run(qc).result()

In [90]:
res_default = backend.run(bdefault_qaoa).result()

In [91]:
custom_unitary = np.asarray(res_custom.results[0].data.unitary)
custom_unitary[np.abs(custom_unitary) < 1e-6] = 0

In [92]:
default_unitary = np.asarray(res_default.results[0].data.unitary)
default_unitary[np.abs(default_unitary) < 1e-6] = 0

In [93]:
np.nonzero(
    np.abs([
        default_unitary[x,x] - custom_unitary[x, sum([2**i * [[int(c) for c in np.binary_repr(x, num_physical_qubits)][num_physical_qubits - 1 - edge_maps[0][ii]] for ii in range(num_physical_qubits)][i] for i in range(num_physical_qubits)])]
    for x in range(2**num_physical_qubits)]) > 1e-6
)

(array([], dtype=int64),)

In [112]:
scores = {i: np.exp(-1j*i) for i in np.linspace(-10,10, 200+1)}
def get_score(x: complex):
    for i in np.linspace(-10,10, 200+1):
        if np.abs(scores[i]- x) < 1e-5:
            return i 
    return x

In [113]:
np.nonzero(
    np.abs([
        evaluate_sparse_pauli_samples([np.binary_repr(x, num_physical_qubits)], h1+h2)[0] 
        - get_score(custom_unitary[x, sum([2**i * [[int(c) for c in np.binary_repr(x, num_physical_qubits)][num_physical_qubits - 1 - edge_maps[0][ii]] for ii in range(num_physical_qubits)][i] for i in range(num_physical_qubits)])])
    for x in range(2**num_physical_qubits)]) > 1e-6
)

(array([], dtype=int64),)

In [105]:
evaluate_sparse_pauli_samples([np.binary_repr(x, num_physical_qubits) for x in range(2**num_physical_qubits)], h1+h2)

array([ 7.5,  6.5, -7.5, -6.5,  7.5,  6.5, -7.5, -6.5, -2.5, -3.5,  2.5,
        3.5, -2.5, -3.5,  2.5,  3.5])

In [106]:
get_score(default_unitary[3,3]), get_score(custom_unitary[3,6])

(np.float64(-6.5), np.float64(-6.5))

In [95]:
np.nonzero(custom_unitary)

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15]),
 array([ 0,  4,  2,  6,  1,  5,  3,  7,  8, 12, 10, 14,  9, 13, 11, 15]))

In [96]:
[(x, sum([2**i * [[int(c) for c in np.binary_repr(x, num_physical_qubits)][num_physical_qubits - 1 - edge_maps[0][ii]] for ii in range(num_physical_qubits)][i] for i in range(num_physical_qubits)]))  for x in range(2**num_physical_qubits)]

[(0, 0),
 (1, 4),
 (2, 2),
 (3, 6),
 (4, 1),
 (5, 5),
 (6, 3),
 (7, 7),
 (8, 8),
 (9, 12),
 (10, 10),
 (11, 14),
 (12, 9),
 (13, 13),
 (14, 11),
 (15, 15)]

In [7]:
hamiltonian = SparsePauliOp(['IIIZ', 'IIZI'], [1, 2])
donor_qc = QuantumCircuit(4)
edge_map = {0:3,1:0,2:1,3:2}
layout = Layout({donor_qc.qubits[key]: val for key, val in edge_map.items()})
new_cost_circ = QuantumCircuit(4)
new_cost_circ.append(PauliEvolutionGate(hamiltonian), [layout.get_virtual_bits()[donor_qc.qubits[i]] for i in range(4)])
new_cost_circ.decompose().draw()

┌───────┐
q_0: ┤ Rz(4) ├
     └───────┘
q_1: ─────────
              
q_2: ─────────
     ┌───────┐
q_3: ┤ Rz(2) ├
     └───────┘